# Joblin — BM25 Lexical Retrieval

Sets up a dedicated **BM25 index** in Elasticsearch for lexical job search — completely separate from the existing semantic (`jobs`) index so neither pipeline interferes with the other.

| Step | What happens |
|------|-------------|
| 1 | Create `jobs_bm25` index with tuned BM25 settings |
| 2 | Bulk-load jobs CSV into the index |
| 3 | Run multi-field BM25 search with field boosting |
| 4 | Filter results by ESCO occupation |
| 5 | Evaluate with nDCG@10 and MRR |

---
> **Note:** This index stores no embeddings — it is purely lexical. Your existing `jobs` vector index is untouched.

## 0 — Config

In [ ]:
CSV_PATH   = "jobs.csv"               # path to your jobs CSV
ES_URL     = "http://localhost:9200"
ES_USER    = "elastic"
ES_PASS    = "iUC5UcRMUKI4JgfEn*rE"
BM25_INDEX = "jobs_bm25"              # kept separate from the vector 'jobs' index

# BM25 hyperparameters (Elasticsearch defaults: k1=1.2, b=0.75)
# k1 — controls term frequency saturation (higher = more TF weight)
# b  — controls length normalisation (1.0 = full normalisation, 0.0 = none)
BM25_K1 = 1.2
BM25_B  = 0.75

## 1 — Imports

In [ ]:
import json
import hashlib
import math

import pandas as pd
from tqdm.notebook import tqdm
from elasticsearch import Elasticsearch, helpers

print("Imports OK")

## 2 — Connect to Elasticsearch

In [ ]:
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASS),
    verify_certs=False,
    ssl_show_warn=False,
)

info = es.info()
print("Connected. ES version:", info["version"]["number"])

## 3 — Create `jobs_bm25` index

We tune the analyzer and set explicit BM25 `k1`/`b` values so the retrieval is reproducible and documented.

In [ ]:
mapping = {
    "settings": {
        "number_of_shards":   1,
        "number_of_replicas": 0,
        "similarity": {
            "joblin_bm25": {
                "type": "BM25",
                "k1":   BM25_K1,
                "b":    BM25_B,
            }
        },
        "analysis": {
            "analyzer": {
                "job_analyzer": {
                    "type":      "custom",
                    "tokenizer": "standard",
                    # lowercase + stop words + stemming for better recall
                    "filter":    ["lowercase", "stop", "porter_stem"],
                }
            }
        }
    },
    "mappings": {
        "properties": {
            # ── Searchable text fields (BM25 applied) ──────────────────────
            "job_title": {
                "type":       "text",
                "analyzer":   "job_analyzer",
                "similarity": "joblin_bm25",
            },
            "job_description": {
                "type":       "text",
                "analyzer":   "job_analyzer",
                "similarity": "joblin_bm25",
            },
            "job_skills": {
                "type":       "text",
                "analyzer":   "job_analyzer",
                "similarity": "joblin_bm25",
            },
            # ── Keyword / filter fields (not tokenised) ────────────────────
            "job_id":       {"type": "keyword"},
            "company":      {"type": "keyword"},
            "job_location": {"type": "keyword"},
            "city":         {"type": "keyword"},
            "country":      {"type": "keyword"},
            "job_level":    {"type": "keyword"},
            "job_type":     {"type": "keyword"},
            # ESCO occupation URI — used for structured filtering
            "esco_occupation_uri": {"type": "keyword"},
        }
    },
}

if es.indices.exists(index=BM25_INDEX):
    es.indices.delete(index=BM25_INDEX)
    print(f"Deleted existing index '{BM25_INDEX}'")

es.indices.create(index=BM25_INDEX, body=mapping)
print(f"Created index '{BM25_INDEX}' with BM25 (k1={BM25_K1}, b={BM25_B})")

## 4 — Load & clean CSV

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")
df.head(3)

In [ ]:
def clean_skills_str(raw) -> str:
    """Return skills as a single space-separated string for BM25 tokenisation."""
    if pd.isna(raw) or str(raw).strip() == "":
        return ""
    if isinstance(raw, list):
        return " ".join(str(s).strip() for s in raw)
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return " ".join(str(s).strip() for s in parsed)
    except (json.JSONDecodeError, TypeError):
        pass
    # comma-separated → space-separated
    return " ".join(s.strip() for s in str(raw).split(",") if s.strip())


def make_job_id(row) -> str:
    link = str(row.get("job_link", "")).strip()
    if link and link != "nan":
        return link
    seed = f"{row.get('job_title','')}|{row.get('company','')}|{row.get('job_location','')}"
    return "job_" + hashlib.md5(seed.encode()).hexdigest()[:12]


df["_job_id"]     = df.apply(make_job_id, axis=1)
df["_skills_str"] = df["job_skills"].apply(clean_skills_str)
df["_title"]      = df["job_title"].fillna("").str.strip()
df["_summary"]    = df["job_summary"].fillna("").str.strip()

before = len(df)
df = df[df["_title"] != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no title. Ready to index: {len(df):,}")

## 5 — Bulk index into `jobs_bm25`

In [ ]:
def generate_actions(dataframe):
    """Yield bulk index actions for elasticsearch.helpers.bulk."""
    for _, row in dataframe.iterrows():
        yield {
            "_index": BM25_INDEX,
            "_id":    row["_job_id"],
            "_source": {
                "job_id":            row["_job_id"],
                "job_title":         row["_title"],
                "job_description":   row["_summary"],
                "job_skills":        row["_skills_str"],
                "company":           str(row.get("company",      "") or ""),
                "job_location":      str(row.get("job_location", "") or ""),
                "city":              str(row.get("city",         "") or ""),
                "country":           str(row.get("country",      "") or ""),
                "job_level":         str(row.get("job_level",    "") or ""),
                "job_type":          str(row.get("job_type",     "") or ""),
                # ESCO URI — populate if you have it, leave blank otherwise
                "esco_occupation_uri": str(row.get("esco_occupation_uri", "") or ""),
            },
        }


success, errors = helpers.bulk(
    es,
    generate_actions(df),
    chunk_size=500,
    raise_on_error=False,
    stats_only=False,
)

print(f"Indexed: {success:,} | Errors: {len(errors)}")
if errors:
    for e in errors[:5]:
        print(" ", e)

In [ ]:
# Confirm document count
es.indices.refresh(index=BM25_INDEX)
count = es.cat.count(index=BM25_INDEX, params={"format": "json"})[0]["count"]
print(f"Documents in '{BM25_INDEX}': {count}")

## 6 — BM25 search function

In [ ]:
def bm25_search(
    query: str,
    top_k: int = 10,
    esco_uri: str | None = None,
    job_level: str | None = None,
    country: str | None = None,
) -> list[dict]:
    """
    Multi-field BM25 search with optional keyword filters.

    Field boost weights:
        job_title^3 — title match is the strongest signal
        job_skills^2 — skills are more specific than description
        job_description^1 — broadest context

    Parameters
    ----------
    query      : free-text search string
    top_k      : number of results to return
    esco_uri   : filter by ESCO occupation URI (exact match)
    job_level  : filter by job level (e.g. 'Senior', 'Junior')
    country    : filter by country keyword
    """
    must_clause = {
        "multi_match": {
            "query": query,
            "fields": [
                "job_title^3",       # most important
                "job_skills^2",      # highly specific
                "job_description",   # broadest
            ],
            "type": "best_fields",   # score = best single-field match
            "tie_breaker": 0.3,      # partial credit for other fields
        }
    }

    # Build filter list
    filters = []
    if esco_uri:
        filters.append({"term": {"esco_occupation_uri": esco_uri}})
    if job_level:
        filters.append({"term": {"job_level": job_level}})
    if country:
        filters.append({"term": {"country": country}})

    es_query = {
        "bool": {
            "must":   must_clause,
            "filter": filters,
        }
    } if filters else must_clause

    response = es.search(
        index=BM25_INDEX,
        query=es_query,
        size=top_k,
        _source=["job_id", "job_title", "company", "job_level", "job_skills", "country"],
    )

    results = []
    for hit in response["hits"]["hits"]:
        results.append({
            "job_id":    hit["_source"]["job_id"],
            "job_title": hit["_source"]["job_title"],
            "company":   hit["_source"].get("company", ""),
            "level":     hit["_source"].get("job_level", ""),
            "country":   hit["_source"].get("country", ""),
            "score":     round(hit["_score"], 4),
        })
    return results

## 7 — Run searches

In [ ]:
# Basic BM25 search
results = bm25_search("python machine learning engineer", top_k=10)

print(f"{'Score':<8} {'Title':<45} {'Company':<25} {'Level'}")
print("-" * 95)
for r in results:
    print(f"{r['score']:<8} {r['job_title'][:43]:<45} {r['company'][:23]:<25} {r['level']}")

In [ ]:
# BM25 search filtered by ESCO occupation URI
# Replace the URI below with one from your normalize endpoint
results_esco = bm25_search(
    query="data engineer spark aws",
    top_k=10,
    esco_uri="http://data.europa.eu/esco/occupation/your-uri-here",  # <-- replace
)

print(f"{'Score':<8} {'Title':<45} {'Company'}")
print("-" * 80)
for r in results_esco:
    print(f"{r['score']:<8} {r['job_title'][:43]:<45} {r['company']}")

In [ ]:
# BM25 filtered by country + level
results_filtered = bm25_search(
    query="software engineer backend",
    top_k=10,
    country="Egypt",     # match your country values in the CSV
    job_level="Senior",  # match your job_level values in the CSV
)

print(f"{'Score':<8} {'Title':<45} {'Country':<15} {'Level'}")
print("-" * 85)
for r in results_filtered:
    print(f"{r['score']:<8} {r['job_title'][:43]:<45} {r['country']:<15} {r['level']}")

## 8 — Evaluate: nDCG@10 and MRR

Define your ground truth as a dict of `{query: [relevant_job_ids]}` — even a small set of 10–20 labelled queries is enough to measure system quality.

In [ ]:
# ── Define your ground truth ──────────────────────────────────────────────────
# Format: { "query string": ["job_id_1", "job_id_2", ...] }
# The first ID in the list is considered most relevant.
GROUND_TRUTH: dict[str, list[str]] = {
    "python data engineer": [],      # <-- fill in relevant job_ids
    "machine learning engineer": [], # <-- fill in relevant job_ids
    "backend developer nodejs": [],  # <-- fill in relevant job_ids
}
# ─────────────────────────────────────────────────────────────────────────────


def dcg(relevances: list[int]) -> float:
    """Compute Discounted Cumulative Gain."""
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances))


def ndcg_at_k(retrieved: list[str], relevant: list[str], k: int = 10) -> float:
    """Compute nDCG@k. Treats all relevant docs as equally relevant (binary)."""
    relevant_set = set(relevant)
    gains = [1 if doc_id in relevant_set else 0 for doc_id in retrieved[:k]]
    ideal = [1] * min(len(relevant_set), k)
    ideal_dcg = dcg(ideal)
    if ideal_dcg == 0:
        return 0.0
    return dcg(gains) / ideal_dcg


def mrr(retrieved: list[str], relevant: list[str]) -> float:
    """Mean Reciprocal Rank — score from the first relevant hit."""
    relevant_set = set(relevant)
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0


K = 10
ndcg_scores, mrr_scores = [], []

print(f"{'Query':<40} {'nDCG@10':>8} {'MRR':>8}")
print("-" * 60)

for query, relevant_ids in GROUND_TRUTH.items():
    if not relevant_ids:
        print(f"{query[:38]:<40} {'(no GT)':>8} {'(no GT)':>8}")
        continue

    results = bm25_search(query, top_k=K)
    retrieved_ids = [r["job_id"] for r in results]

    n = ndcg_at_k(retrieved_ids, relevant_ids, K)
    m = mrr(retrieved_ids, relevant_ids)

    ndcg_scores.append(n)
    mrr_scores.append(m)
    print(f"{query[:38]:<40} {n:>8.4f} {m:>8.4f}")

if ndcg_scores:
    print("-" * 60)
    print(f"{'Mean':<40} {sum(ndcg_scores)/len(ndcg_scores):>8.4f} {sum(mrr_scores)/len(mrr_scores):>8.4f}")

## 9 — Inspect the BM25 explanation

See exactly how Elasticsearch scored a document — useful for debugging and for your paper.

In [ ]:
query = "python data engineer"   # <-- change this

response = es.search(
    index=BM25_INDEX,
    query={
        "multi_match": {
            "query": query,
            "fields": ["job_title^3", "job_skills^2", "job_description"],
            "type": "best_fields",
            "tie_breaker": 0.3,
        }
    },
    size=1,
    explain=True,       # <-- tells ES to return full scoring breakdown
)

top_hit = response["hits"]["hits"][0]
print(f"Top result: {top_hit['_source']['job_title']}")
print(f"Score:      {top_hit['_score']:.4f}")
print("\n--- BM25 explanation ---")
print(json.dumps(top_hit["_explanation"], indent=2))

## 10 — Index stats & settings

In [ ]:
# Document count
count = es.cat.count(index=BM25_INDEX, params={"format": "json"})[0]["count"]
print(f"Documents: {count}")

# Confirm BM25 settings applied
settings = es.indices.get_settings(index=BM25_INDEX)
sim = settings[BM25_INDEX]["settings"]["index"].get("similarity", {})
print(f"BM25 settings: {json.dumps(sim, indent=2)}")